# Fourier Analysis and Convolution

### NEUBEH/PBIO 545 — Quantitative Methods in Neuroscience

*Adapted from* [`matlab/FourierTutorial.m`](../matlab/FourierTutorial.m) by Mike Shadlen and
Adrienne Fairhall.

The Fourier transform is probably the most important transform in applied mathematics. It
takes a function — typically of time or space — and re-expresses it as a function of
**frequency**. The goals of this tutorial are to make you comfortable with what a Fourier
transform *is*, how to compute one, and why it is useful.

Two ideas carry the whole tutorial: the **Fourier transform** itself, and **convolution**.
They turn out to be two faces of the same thing, which is one of the most beautiful results
in applied math and one you will meet constantly in neuroscience — in synaptic filtering, in
receptive fields, in spike-train smoothing, in every piece of recording hardware you will
ever plug into.

A point worth making before anything else: taking a Fourier transform **does not change the
signal**. It re-describes it. If you wanted to describe an electrical trace to a friend over
the phone, you could read out the voltage at each time, or you could read out the amplitude
and phase of each frequency it contains. Same signal, different list of numbers. Sometimes
one list is far more convenient than the other.

| Part | Topic |
|---|---|
| 0 | Conventions and numerical verification — **read this before trusting any spectrum** |
| I | The discrete Fourier transform: definition, frequency axes, transform pairs |
| II | Convolution in the time domain, and the convolution theorem |
| III | Filters, transfer functions, ringing, sampling and aliasing |

Run it cell by cell (**Shift+Enter**). The narrative and the homework questions are part of
the tutorial, not decoration.

**Prerequisites:** the linear algebra tutorial. Comfort with complex numbers helps, but Part
0 and Part I rebuild what you need.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(545)   # fixed seed, so every run reproduces the figures

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.titlesize": 11, "font.size": 9,
})

BLUE, RED, GREEN, PURPLE, GREY = "#3366d9", "#d94d3f", "#33914f", "#6a4fa8", "#808080"

---
## Part 0. Conventions, and a numerical sanity check

This part exists because **spectra are extremely easy to get subtly wrong**, and wrong
spectra look plausible. A peak one bin off, a mirrored axis, a missing factor of $N$ — none
of these throw an error. They just quietly give you the wrong answer.

So before we do anything interesting, we will state the conventions precisely and then
*verify them numerically*. If you are porting your own MATLAB code to Python, this is the
section that will save you.

### The discrete Fourier transform

For a signal $x$ of $N$ samples, indexed $n = 0, 1, \dots, N-1$, the DFT is

$$X_k \;=\; \sum_{n=0}^{N-1} x_n \, e^{-2\pi i \, k n / N}, \qquad k = 0, 1, \dots, N-1$$

and the inverse is

$$x_n \;=\; \frac{1}{N}\sum_{k=0}^{N-1} X_k \, e^{+2\pi i \, k n / N}.$$

Note where the $1/N$ lives: the **forward** transform is unnormalized, the **inverse**
carries the $1/N$. MATLAB's `fft`/`ifft` and NumPy's `np.fft.fft`/`np.fft.ifft` use exactly
this convention, so the *numbers* agree between the two languages. What does **not** agree
is everything around them.

### MATLAB → NumPy: the four things that bite

| | MATLAB | NumPy | Consequence |
|---|---|---|---|
| **Indexing** | `X(1)` is DC | `X[0]` is DC | Every `X(k)` becomes `X[k-1]`. A hand-built axis from `1:N` must become `np.arange(N)`. |
| **Frequency axis** | usually hand-built, e.g. `-nyq:dw:nyq-dw` | `np.fft.fftfreq(N, d=dt)` | Build it with `fftfreq`; use `fftshift` to put it in ascending order. Never transcribe a hand-built axis without checking it. |
| **`conv`** | `conv(a,b)` is `'full'`, length $N_a+N_b-1$ | `np.convolve(a, b, 'full')` — same default | Identical, *but* MATLAB code often slices the result (`s2(1:length(t))`). Reproduce that slice exactly; `mode='same'` is **not** the same slice. |
| **Ordering** | `fftshift` needed to see $-f \dots +f$ | `np.fft.fftshift`, `np.fft.ifftshift` | Identical behaviour. For odd $N$ they are not inverses of each other — use the right one. |

The output of `fft` is ordered as

$$[\,f=0,\; +\Delta f,\; +2\Delta f,\; \dots,\; +f_{\text{Nyq}}\!-\!\Delta f,\;
-f_{\text{Nyq}},\; \dots,\; -\Delta f\,]$$

where $\Delta f = 1/(N\,dt)$ — the positive frequencies first, then the negative ones. This
is why a 10 Hz cosine sampled 100 times per second for 1 s produces peaks at array indices
**10 and 90**, not 10 and $-10$. `fftshift` rotates that into the human-readable order.

### First: build the DFT by hand, once

The fast Fourier transform is a clever $O(N\log N)$ factorization of the sum above. The
algorithm is not the lesson here, so from Part I onward we simply call `np.fft.fft`. But
the *definition* should be concrete before the black box takes over, so we implement the sum
literally, once, for a short signal, and check it agrees.

In [ ]:
def dft_by_hand(x):
    '''Literal transcription of X_k = sum_n x_n exp(-2 pi i k n / N).

    Note the indices: n and k both run 0..N-1, so X[0] is DC. This is the
    Python convention. A MATLAB transcription would run 1..N and need k-1, n-1
    inside the exponent -- which is exactly the off-by-one that ruins ports.
    '''
    x = np.asarray(x, dtype=complex)
    N = x.size
    n = np.arange(N)
    k = n.reshape(N, 1)                  # column of k, row of n -> (N, N) matrix
    W = np.exp(-2j * np.pi * k * n / N)  # the DFT matrix
    return W @ x

x_short = np.array([1.0, 3.0, -2.0, 0.5, 0.0, -1.0, 2.0, 1.5])   # N = 8

X_hand = dft_by_hand(x_short)
X_fft  = np.fft.fft(x_short)

print("k   DFT by hand                     np.fft.fft")
for k in range(x_short.size):
    print(f"{k}   {X_hand[k]: .6f}   {X_fft[k]: .6f}")
print(f"\nmax |difference| = {np.max(np.abs(X_hand - X_fft)):.3e}")

Identical to machine precision. The DFT really is just that matrix multiplication; `fft`
only computes it faster.

### Now the three checks that catch convention errors

**(a) Parseval's theorem.** Energy is conserved by the transform — it is a rotation of the
signal into a new basis, not a distortion of it. With the unnormalized-forward convention,

$$\sum_{n=0}^{N-1} |x_n|^2 \;=\; \frac{1}{N}\sum_{k=0}^{N-1} |X_k|^2 .$$

If you forget the $1/N$ you are off by a factor of $N$ — a factor of 100 in this tutorial.

**(b) The convolution theorem.** Convolution in time is multiplication in frequency:

$$(a * b)_n \;\longleftrightarrow\; A_k \, B_k .$$

But the DFT's multiplication implements **circular** convolution, which wraps around. To
recover the ordinary (linear) convolution that `np.convolve` computes, both sequences must
be zero-padded to at least $N_a + N_b - 1$ before transforming. Forgetting this is the most
common source of mysterious contamination at the beginning of a filtered signal.

**(c) A known frequency lands in a known bin.** A pure sinusoid at exactly $f_0$ Hz, sampled
at $1/dt$ for a whole number of cycles, must put all of its weight in bins $k = f_0 N dt$ and
$N - f_0 N dt$ and nowhere else.

In [ ]:
# ---------- (a) Parseval's theorem ----------
x = rng.standard_normal(100)
X = np.fft.fft(x)
energy_time = np.sum(np.abs(x) ** 2)
energy_freq = np.sum(np.abs(X) ** 2) / x.size

print("(a) PARSEVAL")
print(f"    sum |x[n]|^2            = {energy_time:.10f}")
print(f"    (1/N) sum |X[k]|^2      = {energy_freq:.10f}")
print(f"    relative error          = {abs(energy_time - energy_freq) / energy_time:.3e}")
print(f"    (without the 1/N it would be {np.sum(np.abs(X)**2):.4f} -- off by exactly N = {x.size})")

# ---------- (b) convolution theorem ----------
a = np.exp(-np.arange(20) / 3.0)          # 20-sample kernel
b = np.sin(np.arange(15) / 2.0)           # 15-sample signal
L = a.size + b.size - 1                   # length of the FULL linear convolution

conv_direct   = np.convolve(a, b, mode="full")
conv_viafft   = np.fft.ifft(np.fft.fft(a, L) * np.fft.fft(b, L)).real
conv_nopadding = np.fft.ifft(np.fft.fft(a) * np.fft.fft(b, a.size)).real  # WRONG: circular

print("\n(b) CONVOLUTION THEOREM")
print(f"    len(np.convolve(a, b, 'full'))          = {conv_direct.size}  (= Na + Nb - 1 = {L})")
print(f"    max |np.convolve  -  ifft(fft*fft), padded to {L}| = "
      f"{np.max(np.abs(conv_direct - conv_viafft)):.3e}")
wrap = np.max(np.abs(conv_direct[:a.size] - conv_nopadding))
print(f"    max |error, UNPADDED (circular)|        = {wrap:.4f}  <-- wrap-around")
print(f"    ... as a fraction of the signal itself  = {wrap / np.max(np.abs(conv_direct)):.1%}"
      f", all of it in the first few samples")

# ---------- (c) a known frequency in a known bin ----------
dt_chk, N_chk, f0 = 0.01, 100, 10.0        # 10 Hz, 100 Hz sampling, 1 s of data
t_chk = np.arange(N_chk) * dt_chk
X_chk = np.fft.fft(np.cos(2 * np.pi * f0 * t_chk))
freqs = np.fft.fftfreq(N_chk, d=dt_chk)
peaks = np.flatnonzero(np.abs(X_chk) > 1e-6)

print("\n(c) KNOWN FREQUENCY -> KNOWN BIN")
print(f"    bins with non-negligible power: {peaks.tolist()}")
print(f"    frequencies at those bins:      {freqs[peaks].tolist()} Hz")
print(f"    values there:                   {np.round(X_chk[peaks], 6).tolist()}")
print(f"    expected value N/2 = {N_chk / 2:g} at each of +{f0:g} and -{f0:g} Hz")

All three pass. Read the numbers, not just the absence of an error message:

- Parseval matches to $\sim 10^{-16}$ relative — and note the printed reminder of what the
  answer would have been without the $1/N$.
- The zero-padded FFT product reproduces `np.convolve` to $\sim 10^{-15}$. The unpadded
  version is wrong by a few tenths of a percent here — small only because this kernel decays
  quickly — and that error is entirely concentrated at the **start** of the output, where the
  tail of the convolution has wrapped around. With a slowly decaying kernel it is not small
  at all; Part II.3 shows a case where the wrap-around is as large as the signal.
- A 10 Hz cosine puts weight in exactly two bins, at $+10$ and $-10$ Hz, each with value
  $N/2 = 50$ and zero imaginary part. That is $\cos\theta = \tfrac12(e^{i\theta} +
  e^{-i\theta})$ made visible.

If any of these ever fails in your own code, stop and fix the axis or the normalization
before interpreting a single plot.

---
## Part I. The discrete Fourier transform

### I.1 Discrete signals

Everything here is numerical, so every function is **discrete**: time takes integer values
multiplied by a scaling factor $dt$. This is not a compromise for the sake of the computer —
it is what real data always looks like. You sample at some finite rate, and that is all you
have.

In [ ]:
dt = 0.01                     # seconds per sample
tmax = 1.0                    # seconds
t = np.arange(0, tmax, dt)    # MATLAB: t = [0:dt:tmax-dt]'
N = t.size

samplingRate = 1 / dt         # Hz

print(f"dt           = {dt} s")
print(f"N            = {N} samples")
print(f"samplingRate = {samplingRate:g} Hz")
print(f"t[0] = {t[0]:g},  t[-1] = {t[-1]:g}")

Think of a discrete function as a series of **weights**, one at each point in $t$. A Gaussian
bump centred at 0.5 s makes the point: matplotlib will happily draw a smooth curve through
the samples, but all we actually possess is the heights.

In [ ]:
f1 = np.exp(-0.5 * (t - 0.5) ** 2 / 0.05 ** 2)
gaus1 = f1

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t, f1, color=GREY, lw=1, label="interpolated (a lie matplotlib tells you)")
ax.stem(t, f1, linefmt=BLUE, markerfmt="o", basefmt=" ", label="the actual samples")
ax.set(xlabel="Time (s)", ylabel="Amplitude", title="A discrete Gaussian: 100 weights, not a curve")
ax.legend(fontsize=8)
fig.tight_layout()

### I.2 A menagerie of signals

We now define seven signals that will follow us through the whole tutorial. Each one is a
list of 100 weights; each has an instructive Fourier transform.

In [ ]:
# f1: Gaussian, centred at 0.5 s                              (already defined above)

# f2: pulse
f2 = np.zeros_like(t)
f2[(t > 0.45) & (t <= 0.55)] = 1.0
pulse1 = f2
print(f"pulse: {int(f2.sum())} samples wide, from t = {t[f2 > 0][0]:.2f} to {t[f2 > 0][-1]:.2f} s")

# f3: sine at 1 Hz, phase-referenced to t = 0.5
f3 = np.sin(2 * np.pi * 1 * (t - 0.5))
sin1 = f3

# f4: cosine at 10 Hz, phase-referenced to t = 0.5
f4 = np.cos(2 * np.pi * 10 * (t - 0.5))
cos10 = f4

# f5: causal exponential, switched on at 0.5 s
tau = 0.1
f5 = np.exp(-(t - 0.5) / tau)
f5[t < 0.5] = 0.0
exp1 = f5

# f6: discrete delta function at 0.5 s
f6 = np.zeros_like(t)
f6[np.flatnonzero(t >= 0.5)[0]] = 1.0      # MATLAB: f6(min(find(t>=.5))) = 1
delt1 = f6

# f7: comb (strobe) at 10 Hz -- a tooth every 0.1 s
#
# GOTCHA: the MATLAB original writes  f7 = mod(t,.1)==0 .  In floating point that is
# NOT what you want:
print("teeth found by the literal 'mod(t,0.1)==0' test:",
      np.flatnonzero(np.mod(t, 0.1) == 0).tolist(), " <-- should be 0,10,20,...,90")
# so we build the comb by index arithmetic instead, which is exact:
f7 = np.zeros_like(t)
f7[::10] = 1.0
comb10 = f7
print("teeth in the comb we actually use:      ", np.flatnonzero(f7).tolist())

Look at what that `mod` test returned. Only **five** of the ten intended teeth survive,
scattered at 0, 10, 20, 40 and 80 — because `0.3 % 0.1` in double precision is
$0.0999\ldots$, not zero. The MATLAB original has this bug too; its "comb" is a ragged,
non-periodic set of spikes, and any spectrum computed from it is not the spectrum of a comb.

This is the general lesson: **never test floating-point values for equality**. Build regular
grids by index arithmetic (`f7[::10] = 1`) or compare with a tolerance
(`np.isclose`). We use the index version from here on, so our comb really is periodic.

In [ ]:
sigs = [(f1, "f1: Gaussian"), (f2, "f2: pulse"), (f3, "f3: sin, 1 Hz"),
        (f4, "f4: cos, 10 Hz"), (f5, "f5: exponential"), (f6, "f6: delta"),
        (f7, "f7: comb, 10 Hz")]

fig, axes = plt.subplots(7, 1, figsize=(7, 10), sharex=True)
for ax, (s, name) in zip(axes, sigs):
    if name.startswith(("f6", "f7", "f2")):
        ax.stem(t, s, linefmt=BLUE, markerfmt=".", basefmt=" ")
    else:
        ax.plot(t, s, color=BLUE, lw=1.5)
    ax.set_ylabel(name, fontsize=8, rotation=0, ha="right", va="center")
axes[-1].set_xlabel("Time (s)")
axes[0].set_title("Seven discrete signals, all 100 samples long")
fig.tight_layout()

A reflection point — and you have not yet seen a single Fourier transform.

Every one of these signals is a **sum of weighted delta functions**. The comb makes that
obvious, but it is equally true of the Gaussian: it is a value at $t=0$, plus a value at
$t=dt$, plus a value at $t=2dt$, and so on. A discrete signal *is* a continuous function
multiplied by a comb whose spacing is $dt$. We will come back to that in Part III, where it
turns out to explain aliasing.

Now recall the idea of a **basis set** from linear algebra. A coordinate frame is a set of
vectors spanning a space. Our 100-sample signal is a point in a 100-dimensional space, and
the coordinate system we have been using without noticing is the set of delta functions —
$[1,0,0,\dots]$, $[0,1,0,\dots]$, and so on. Axis 1 asks "how strong was the signal at
$t=0$?", axis 2 asks "how strong at $t=dt$?". These axes are orthogonal: the delta function
at time $t$ has zero dot product with the delta function at $t' \ne t$.

The Fourier transform is nothing more exotic than **a different choice of orthogonal axes**
for that same 100-dimensional space — axes that are sines and cosines rather than delta
functions. Rotating a vector into new axes does not change the vector.

### I.3 Why the basis functions are complex

The continuous Fourier transform of $f(t)$ is

$$F(\omega) \;=\; \frac{1}{\sqrt{2\pi}}\int f(t)\, e^{-i\omega t}\, dt .$$

The integral is the continuous version of a dot product: we are **projecting** the signal
onto the basis function $e^{-i\omega t}$, one basis function for every frequency $\omega$.

So where are the sines and cosines? Euler's identity:

$$e^{ix} \;=\; \cos x + i \sin x .$$

Every complex number is really two-dimensional, with a real axis and an imaginary axis. The
number $e^{ix}$ has length $\cos x$ along the real axis and $\sin x$ along the imaginary
axis, and total length $\sqrt{\cos^2 x + \sin^2 x} = 1$. Scaling by an amplitude $A$ gives
$A e^{ix}$, a vector of length $A$ at angle $x$.

In [ ]:
for x in [0.5, np.pi, -np.pi, np.pi / 2]:
    z = np.exp(1j * x)
    print(f"exp(i*{x: .6f}) = {z.real: .6f} {'+' if z.imag >= 0 else '-'} {abs(z.imag):.6f}i"
          f"   |z| = {abs(z):.6f}")

(In MATLAB you occasionally have to `clear i` first, because `i` is a perfectly legal
variable name that people overwrite. Python spells the imaginary unit `1j` — a numeric
literal, not an identifier — so it cannot be shadowed. One fewer thing to go wrong.)

So every $e^{-i\omega t}$ delivers **two** numbers per frequency: a real one, the weight on
the cosine, and an imaginary one, the weight on the sine. Why do we need two? Because a
frequency component is specified by both an **amplitude** and a **phase**. Recall

$$\sin(x + \phi) \;=\; \cos\phi \,\sin x \;+\; \sin\phi\, \cos x,$$

so a sinusoid of amplitude $a$ and phase $\phi$ is $a\cos\phi$ of a sine plus $a\sin\phi$ of
a cosine, and its amplitude is $a\sqrt{\cos^2\phi + \sin^2\phi} = a$. Having both a sine and
a cosine at each frequency is precisely what lets you build an arbitrary phase. Complex
numbers are just a tidy way to carry the pair.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 4), sharex=True)
axes[0].plot(t, np.real(np.exp(1j * 2 * np.pi * t)), color=BLUE, lw=1.5)
axes[0].set(ylabel="real part", title=r"$e^{i\omega t}$ at $\omega = 2\pi$  (i.e. $f$ = 1 Hz)")
axes[1].plot(t, np.imag(np.exp(1j * 2 * np.pi * t)), color=RED, lw=1.5)
axes[1].set(xlabel="t (s)", ylabel="imaginary part")
for a in axes:
    a.set_ylim(-1.15, 1.15)
fig.tight_layout()

A note on symbols that trips people up constantly: $\omega$ is **angular** frequency, in
radians per second, and $f$ is frequency in **hertz**, cycles per second. They differ by
$\omega = 2\pi f$. Throughout this tutorial we factor the $2\pi$ out and work in hertz,
because that is what the frequency axis of an FFT is naturally labelled in.

### Phase shifts are multiplication by $e^{i\phi}$

Multiplying a complex sinusoid by $e^{i\phi}$ rotates it in the complex plane, which shifts
the wave in time. The original tutorial animates this; here are six frozen frames, with the
polar representation of $e^{i\phi}$ beneath each one.

In [ ]:
phis = np.linspace(0, 2 * np.pi, 7)[:6]      # 0, 60, ..., 300 degrees

fig = plt.figure(figsize=(10, 4.2))
for j, phi in enumerate(phis):
    ax = fig.add_subplot(2, 6, j + 1)
    ax.plot(t, np.cos(2 * np.pi * t), color=GREY, lw=1)
    ax.plot(t, np.real(np.exp(1j * (2 * np.pi * t + phi))), color=RED, lw=1.8)
    ax.set_ylim(-1.2, 1.2)
    ax.set_xticks([0, 0.5, 1]); ax.set_yticks([])
    ax.set_title(rf"$\phi = {np.degrees(phi):.0f}^\circ$", fontsize=9)
    if j: ax.set_xticklabels([])

    pol = fig.add_subplot(2, 6, j + 7, projection="polar")
    pol.plot([0, phi], [0, 1], color=RED, lw=2)
    pol.plot([phi], [1], "o", color=RED, ms=5)
    pol.set_yticklabels([]); pol.set_xticklabels([]); pol.set_ylim(0, 1)
    pol.grid(alpha=0.3)
fig.suptitle(r"Multiplying by $e^{i\phi}$ shifts the wave; grey is the unshifted cosine", y=1.0)
fig.tight_layout()

So: projecting onto $e^{-i\omega t}$ projects onto a sine **and** a cosine of frequency
$\omega$ at once. Sines and cosines of different frequencies are orthogonal, so this is a
genuine orthogonal basis — a rotation of the 100-dimensional space, nothing more. Sines and
cosines are not the only possible alternative basis (wavelets and principal components are
others you will meet), but they are the one that comes up most.

### I.4 The first FFT, and the first surprise

Let's transform a 10 Hz cosine and plot the result naively, the way you would if you had not
thought about it.

In [ ]:
y = np.cos(2 * np.pi * 10 * t)
fcos = np.fft.fft(y)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(t, y, color=BLUE, lw=1.5)
axes[0].set(xlabel="Time (s)", ylabel="Amplitude", title="y = cos(2$\\pi\\,$10$\\,t$)")
axes[1].plot(fcos.real, fcos.imag, color=RED, lw=1, marker="o", ms=3)
axes[1].set(xlabel="real part", ylabel="imaginary part",
            title="plot(fft(y)) -- what did we just draw?")
fig.tight_layout()

print(f"fcos.dtype = {fcos.dtype},  fcos.shape = {fcos.shape}")

What the *$&#^%? (Pardon my Australian.)

The FFT of a real signal is a **complex** array, and plotting a complex array plots the real
part against the imaginary part — an Argand diagram, not a spectrum. MATLAB's `plot` does
exactly the same thing, which is how generations of students have met this figure. Let's
separate the two parts and plot each against the array index.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
axes[0].plot(np.arange(N), fcos.real, color=BLUE, lw=1.5)
axes[0].set(xlabel="array index k", ylabel="value", title="Real part of fft(cos)")
axes[1].plot(np.arange(N), fcos.imag, color=RED, lw=1.5)
axes[1].set(xlabel="array index k", title="Imaginary part of fft(cos)")
fig.tight_layout()

big = np.flatnonzero(np.abs(fcos) > 1e-8)
print(f"indices with non-negligible amplitude: {big.tolist()}")
print(f"values there: {np.round(fcos[big], 8).tolist()}")
print(f"largest |imaginary part| anywhere:     {np.max(np.abs(fcos.imag)):.3e}  (i.e. zero)")

Now we are getting somewhere. The real part has two spikes; the imaginary part is zero to
within numerical noise ($\sim 10^{-14}$).

Two questions. **Why two spikes, and why at indices 10 and 90?**

For "why two", recall one more piece of high-school algebra:

$$\cos a = \frac{e^{ia} + e^{-ia}}{2}, \qquad \sin a = \frac{e^{ia} - e^{-ia}}{2i}.$$

The transform scans over $\omega$ looking for non-zero projections. Under the integral sign
we have

$$e^{-i\omega t}\left[e^{i 2\pi 10 t} + e^{-i 2\pi 10 t}\right]
= e^{i(2\pi 10 - \omega)t} + e^{-i(2\pi 10 + \omega)t}.$$

The integral of a complex exponential over a whole number of cycles is zero *unless the
exponent cancels exactly*, leaving $e^0 = 1$. That happens for the first term at
$\omega = 2\pi\cdot 10$ and for the second at $\omega = -2\pi \cdot 10$. So we expect two
peaks, at $f = +10$ and $f = -10$ Hz — and each carries half the signal, hence the printed
value $N/2 = 50$.

For "why 10 and 90": as stated in Part 0, `fft` returns the positive frequencies first and
then the negative ones. Index 90 out of 100 *is* $-10$ Hz. `fftshift` rearranges the array
into ascending frequency order, and from here on we always use it.

### I.5 What are the highest and lowest frequencies?

The **lowest** non-zero frequency is set by the length of the record. If you observe for $T$
seconds you cannot resolve anything slower than one cycle in $T$, so $\Delta f = 1/T$. Here
$T = 1$ s, so the bins are 1 Hz apart.

The **highest** is set by the sampling interval. You need at least two samples per cycle —
one on the way up and one on the way down — so the highest representable frequency is
$1/(2\,dt)$. This is the **Nyquist frequency**. Sample any slower than twice the highest
frequency present and you will not merely lose that frequency; it will masquerade as a lower
one. That is aliasing, and Part III demonstrates it.

In [ ]:
nyq = samplingRate / 2      # Nyquist frequency, Hz
dw  = 1 / tmax              # frequency resolution, Hz

# MATLAB original built the axis by hand:  fax = -nyq : dw : nyq-dw
# In Python, build it from fftfreq so it cannot drift out of step with fft():
fax = np.fft.fftshift(np.fft.fftfreq(N, d=dt))

fax_byhand = np.arange(-nyq, nyq, dw)      # the literal MATLAB transcription
print(f"nyq = {nyq:g} Hz,  df = {dw:g} Hz,  {N} bins")
print(f"fax[:4]  = {fax[:4]}   fax[-4:] = {fax[-4:]}")
print(f"fax[N//2] = {fax[N // 2]:g}  <-- DC sits here after fftshift, at index N//2 = {N // 2}")
print(f"agrees with the hand-built MATLAB axis: {np.array_equal(fax, fax_byhand)}")

Here the hand-built axis and `fftfreq` happen to agree exactly, because $N$ is even and the
arithmetic is clean. **Do not rely on that.** For odd $N$, or for $dt$ that does not divide
neatly, the hand-built version drifts or comes out the wrong length, and you get a spectrum
that is shifted by a bin with no warning. `np.fft.fftfreq(N, d=dt)` is defined to match
`np.fft.fft(x)` bin for bin, always. Use it.

Now the same two transforms, properly labelled, and with the sine for comparison. Both panels
in a row share y-limits so the real and imaginary parts are genuinely comparable — otherwise
matplotlib autoscales the numerical-noise panel and you see a spectacular-looking spectrum of
$10^{-14}$.

In [ ]:
ysin = np.sin(2 * np.pi * 10 * t)
fsin = np.fft.fft(ysin)

fig, axes = plt.subplots(2, 2, figsize=(9, 5.5), sharex=True, sharey=True)
panels = [(fcos, "cos", 0), (fsin, "sin", 1)]
for F, nm, row in panels:
    axes[row, 0].stem(fax, np.fft.fftshift(F.real), linefmt=BLUE, markerfmt=".", basefmt=" ")
    axes[row, 0].set_ylabel(f"Real part of ft({nm})")
    axes[row, 1].stem(fax, np.fft.fftshift(F.imag), linefmt=RED, markerfmt=".", basefmt=" ")
    axes[row, 1].set_ylabel(f"Imag part of ft({nm})")
axes[0, 0].set_ylim(-55, 55)
for a in axes[1]:
    a.set_xlabel("Frequency (Hz)")
fig.suptitle("Transforms of a 10 Hz cosine (top) and a 10 Hz sine (bottom)")
fig.tight_layout()

print(f"cos: real part at +10 Hz = {fcos[10].real:+.1f},  at -10 Hz = {fcos[90].real:+.1f}")
print(f"sin: imag part at +10 Hz = {fsin[10].imag:+.1f},  at -10 Hz = {fsin[90].imag:+.1f}")

Exactly as expected. The cosine's weight is entirely **real** and **even** — the same value
$+50$ at $+10$ and $-10$ Hz. The sine's weight is entirely **imaginary** and **odd** —
$-50$ at $+10$ Hz and $+50$ at $-10$ Hz.

The minus sign on the positive-frequency component of the sine surprises people. It comes
straight from $\sin a = (e^{ia} - e^{-ia})/2i$: dividing by $i$ is multiplying by $-i$, so
the $+f$ term picks up a factor of $-i/2$ and the $-f$ term $+i/2$.

> ### Homework question 1
> **(a)** Predict, without running anything, the real and imaginary parts of
> `np.fft.fft(np.cos(2*np.pi*10*t + np.pi/4))`. Then check. Where did the amplitude go?
>
> **(b)** What happens to the transform of a 10 Hz cosine if you change `tmax` to 1.5 s,
> leaving `dt` alone? The signal no longer contains a whole number of cycles in the record.
> Plot the amplitude spectrum and explain the smear. (This is **spectral leakage**, and it is
> why people apply windows to data.)
>
> **(c)** What is the transform of a constant signal, `np.ones_like(t)`? Of
> `np.cos(2*np.pi*50*t)` — a cosine exactly at Nyquist? Why is the second one strange?

### I.6 A sum of sinusoids

Now something slightly less trivial: three sines added together. The time-domain waveform is
already vaguely square-wave-like; its transform makes the recipe explicit. Because all three
components are sines, we only need the imaginary part.

In [ ]:
y3 = np.sin(2 * np.pi * 3 * t) + 0.33 * np.sin(2 * np.pi * 9 * t) + 0.2 * np.sin(2 * np.pi * 15 * t)
Y3 = np.fft.fft(y3)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2))
axes[0].plot(t, y3, color=BLUE, lw=1.5)
axes[0].set(xlabel="Time (s)", ylabel="Amplitude", title="sum of 3 sines: 3, 9 and 15 Hz")
axes[1].stem(fax, np.fft.fftshift(Y3.imag), linefmt=RED, markerfmt=".", basefmt=" ")
axes[1].set(xlabel="Frequency (Hz)", ylabel="Imaginary part of FT",
            title="Imaginary part of the transform")
fig.tight_layout()

for f0, amp in [(3, 1.0), (9, 0.33), (15, 0.2)]:
    k = int(f0 * tmax)
    print(f"{f0:2d} Hz: FT imag = {Y3[k].imag:+8.3f}   expected -amp*N/2 = {-amp * N / 2:+8.3f}")
print(f"largest |real part| anywhere: {np.max(np.abs(Y3.real)):.3e}")

Three peaks, at exactly 3, 9 and 15 Hz (and their negative mirrors), with heights $-50$,
$-16.5$ and $-10$ — that is $-\text{amplitude} \times N/2$ in each case. The real part is
zero everywhere to within $10^{-13}$, because every component is a pure sine.

### I.7 A gallery of transform pairs

Now the seven signals from I.2, each with its **amplitude spectrum** $|X_k|$. Amplitude
throws away phase, which is the right first thing to look at.

In [ ]:
fig, axes = plt.subplots(7, 2, figsize=(9.5, 12))
for row, (s, name) in enumerate(sigs):
    A = np.fft.fftshift(np.abs(np.fft.fft(s)))
    discrete = name.startswith(("f2", "f6", "f7"))
    if discrete:
        axes[row, 0].stem(t, s, linefmt=BLUE, markerfmt=".", basefmt=" ")
    else:
        axes[row, 0].plot(t, s, color=BLUE, lw=1.3)
    axes[row, 0].set_ylabel(name, fontsize=8, rotation=0, ha="right", va="center")
    if name.startswith(("f3", "f4", "f6", "f7")):
        axes[row, 1].stem(fax, A, linefmt=PURPLE, markerfmt=".", basefmt=" ")
    else:
        axes[row, 1].plot(fax, A, color=PURPLE, lw=1.3)
    axes[row, 1].set_xlim(-nyq, nyq)
    for c in (0, 1):
        if row < 6:
            axes[row, c].set_xticklabels([])
axes[6, 0].set_xlabel("Time (s)")
axes[6, 1].set_xlabel("Frequency (Hz)")
axes[0, 0].set_title("signal  s(t)")
axes[0, 1].set_title("amplitude spectrum  |S(f)|")
fig.tight_layout()

Read the seven rows one at a time. Every one of these pairs is worth committing to memory.

1. **Gaussian $\leftrightarrow$ Gaussian.** A Gaussian in time transforms to a Gaussian in
   frequency. This is a unique property of the Gaussian, and the widths are *reciprocal*:
   narrow in time means wide in frequency, and vice versa. This is the same mathematics as
   the Heisenberg uncertainty principle. Something localized in time is spread out in
   spectrum.

2. **Pulse $\leftrightarrow$ sinc.** A rectangular pulse transforms to
   $\sin(\pi f T)/(\pi f T)$ — a decaying oscillation with regularly spaced zeros. This
   matters enormously in practice: *any* data cut out of a longer record with a hard edge has
   been multiplied by a rectangle, and therefore has this oscillation convolved into its
   spectrum. The phenomenon is called **ringing**, and Part III returns to it.

3. **Sine at 1 Hz $\leftrightarrow$ a pair of spikes at $\pm 1$ Hz.** Exactly as derived.

4. **Cosine at 10 Hz $\leftrightarrow$ a pair of spikes at $\pm 10$ Hz.**

5. **Exponential $\leftrightarrow$ Lorentzian-ish $1/f$ decay.** The amplitude falls off as
   $1/\sqrt{1 + (2\pi f\tau)^2}$, i.e. roughly as $1/f$ at high frequency. An exponential
   decay is a low-pass filter, which is why the RC circuit is the canonical one.

6. **Delta $\leftrightarrow$ flat.** A single impulse is built by adding sinusoids of
   **equal amplitude at every frequency**. Only their phases distinguish an impulse at
   $t=0.5$ from one at $t=0$. This is the deepest of the seven and the reason impulse
   responses are so useful (Part III).

7. **Comb $\leftrightarrow$ comb.** A 10 Hz comb in time transforms to a comb in frequency
   with teeth every 10 Hz. Reciprocal spacing again: sample more finely in time, and the
   frequency-domain teeth move further apart. This one *is* aliasing, in disguise.

### I.8 Real, imaginary and amplitude — and why we centre the signals first

Amplitude spectra hide phase. To see the real and imaginary parts cleanly, we play one trick:
before transforming, we circularly shift each signal so that its **centre lands at index 0**.

Why? Because the DFT treats index 0 as $t=0$. Our signals are built around $t = 0.5$ s, so
they all carry a half-record time shift, and a time shift multiplies the transform by
$e^{-2\pi i f t_0}$ — a phase ramp that scrambles real and imaginary parts into each other.
Removing it lets the even/odd structure show through.

`np.fft.ifftshift` does exactly this (it moves index `N//2` to index 0), and it is what the
MATLAB original calls `ifftshift` too. Note the asymmetry: use **`ifftshift` before** the
transform to centre a signal, and **`fftshift` after** the transform to order the frequency
axis. For even $N$ they happen to be the same permutation; for odd $N$ they are not, and
swapping them costs you one sample of shift.

In [ ]:
fig, axes = plt.subplots(7, 4, figsize=(11.5, 12.5))
for row, (s, name) in enumerate(sigs):
    a = np.fft.fft(np.fft.ifftshift(s))          # centre the signal first
    lim = 1.1 * np.max(np.abs(a))
    axes[row, 0].plot(t - 0.5, s, color=BLUE, lw=1.2)
    axes[row, 0].set_ylabel(name, fontsize=8, rotation=0, ha="right", va="center")
    for col, (vals, colr) in enumerate([(np.real(a), GREEN), (np.imag(a), RED),
                                        (np.abs(a), PURPLE)], start=1):
        axes[row, col].plot(fax, np.fft.fftshift(vals), color=colr, lw=1.2)
        axes[row, col].set_ylim(-lim, lim)       # all three share limits: comparable
        axes[row, col].set_xlim(-nyq, nyq)
    for c in range(4):
        if row < 6:
            axes[row, c].set_xticklabels([])
axes[6, 0].set_xlabel("Time relative to centre (s)")
for c in range(1, 4):
    axes[6, c].set_xlabel("Frequency (Hz)")
for c, ttl in enumerate(["s(t), centred", "Real part of FT", "Imaginary part of FT",
                         "Amplitude of FT"]):
    axes[0, c].set_title(ttl)
fig.tight_layout()

Pause and reflect; there is a lot in that figure.

**Zero frequency is special.** It sits at the centre of each spectrum and represents the mean
level — the $\cos(0)$ component. It is always purely real, because $\sin(0) = 0$.

**Even and odd signals separate cleanly.** The centred Gaussian, pulse, cosine, delta and
comb are all *even* about their centre, and their transforms are purely real (the imaginary
panels are flat at zero). The centred sine is *odd*, and its transform is purely imaginary.
The exponential is neither, and has both.

Two small caveats visible in the figure, worth noticing rather than glossing over:

- The **pulse** (row 2) is not perfectly even. `t > 0.45 & t <= 0.55` selects the ten samples
  from 0.46 to 0.55 s, whose centre of mass is half a sample to the right of $t = 0.5$. So a
  tiny residual phase ramp survives and the imaginary panel is not quite flat. A genuinely
  even pulse needs an odd number of samples.
- The **exponential** (row 5) is causal — it starts at the centre and decays to the right. It
  cannot be even or odd, and both parts are populated.

### I.9 Invertibility, and counting degrees of freedom

The transform is invertible: from the list of complex weights we can rebuild the signal. That
only works if we kept enough numbers, so let's count.

We have $N = 100$ time samples. The highest frequency is Nyquist $= 50$ Hz and the resolution
is 1 Hz, so there are 50 distinct positive frequencies. How do 100 numbers become 50
frequencies? Because each frequency needs **two** numbers, a cosine and a sine weight. Except:

- at $f = 0$ there is only one, since $\sin 0 = 0$ everywhere;
- at $f = f_{\text{Nyq}}$ there is again only one, because with two samples per cycle we can
  represent alternating $\pm$ values (a cosine) but a sine sampled that way is identically
  zero.

So: two components each for $f = 1 \dots 49$ Hz, plus one at DC, plus one at Nyquist
$= 2 \times 49 + 2 = 100$. Exactly the number of time samples. The bookkeeping works.

Then what about the negative frequencies, which appear to double the count back to 200? The
answer is **Hermitian symmetry**. For a *real* signal,

$$X_{-k} = \overline{X_{k}} \qquad\text{(equivalently } X_{N-k} = \overline{X_k}\text{)},$$

so the negative half is completely determined by the positive half: amplitude is even in
frequency, phase is odd. The transform genuinely has 100 free real numbers, not 200. (The FFT
returns all 200 because it is written to handle complex-valued inputs too, where the
redundancy is absent.)

In [ ]:
X5 = np.fft.fft(f5)                       # the exponential: neither even nor odd
herm = np.max(np.abs(X5[1:] - np.conj(X5[1:][::-1])))
print(f"max |X[k] - conj(X[N-k])| over k = 1..N-1 : {herm:.3e}   (Hermitian symmetry)")

# Because of that redundancy, rfft stores only the non-redundant half.
R5 = np.fft.rfft(f5)
rfax = np.fft.rfftfreq(N, d=dt)
print(f"fft  returns {X5.size} complex numbers, covering {fax[0]:g} .. {fax[-1]:g} Hz")
print(f"rfft returns {R5.size} complex numbers, covering {rfax[0]:g} .. {rfax[-1]:g} Hz"
      f"   (= N/2 + 1)")
print(f"rfft agrees with the first half of fft: "
      f"{np.allclose(R5, X5[:R5.size])}")
print(f"real-number count: 2*{R5.size} - 2 (DC and Nyquist have no imaginary part) = "
      f"{2 * R5.size - 2} = N = {N}")

# and the round trip
print(f"\nmax |ifft(fft(f5)) - f5| = {np.max(np.abs(np.fft.ifft(X5) - f5)):.3e}")
print(f"max |irfft(rfft(f5)) - f5| = {np.max(np.abs(np.fft.irfft(R5, n=N) - f5)):.3e}")

**When should you use `rfft` instead of `fft`?** Use `rfft` when your signal is real and you
only want the physically distinct half of the spectrum — it is twice as fast, uses half the
memory, and makes it impossible to accidentally double-count power. Use `fft` when you want
to *see* the symmetry (as in the gallery figures above), when the signal is complex, or when
you are going to manipulate the spectrum and transform back and want the bookkeeping to be
transparent.

This tutorial mostly uses `fft` + `fftshift`, because seeing both halves is pedagogically the
point. Where we compute a power spectrum in Part III, we use `rfft` and say so.

> ### Homework question 2
> **(a)** Which of the seven signals in the gallery are **even** ($f(x) = f(-x)$, about their
> centre)? Which is **odd** ($f(x) = -f(-x)$)? Which is neither? Now state the rule
> connecting each of the three categories to its Fourier transform.
>
> **(b)** Verify your rule numerically for one signal of each type, using
> `np.max(np.abs(...))` rather than eyeballing a plot.
>
> **(c)** The Gaussian `f1` has SD 0.05 s. Compute the SD of its amplitude spectrum by fitting
> or by moments, for SDs of 0.02, 0.05 and 0.1 s. Confirm the product of the two widths is
> roughly constant, and work out what the constant should be.

*Answer to 2(a), for checking after you have tried it:* even $\rightarrow$ purely real
transform; odd $\rightarrow$ purely imaginary; neither $\rightarrow$ both. (Even and odd here
mean about the signal's own centre, which is why we applied `ifftshift` first.)

---
## Part II. Convolution

One of the most important uses of the Fourier transform is computing **convolution**.
Convolution takes a signal and turns it into a new signal — usually a filtered, blurred,
smeared version of the old one. We will build the intuition twice: once purely in the time
domain, and once in the frequency domain. Keep the frequency picture in the back of your mind
while we do the time-domain version, and ask yourself: if blurring smooths out sharp
features, isn't that the same as attenuating high frequencies?

### II.1 Convolution in the time domain

Start with the simplest possible signal — a delta function. Call it a click, if it is sound
pressure, or a thin line, if it is light intensity along a row of pixels.

In [ ]:
timeOfImpulse = 0.1
s1_imp = np.zeros_like(t)
s1_imp[np.argmin(np.abs(t - timeOfImpulse))] = 1.0

tau_b = 0.03                       # blur time constant
b = np.exp(-t / tau_b)             # causal exponential blur kernel

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].stem(t, s1_imp, linefmt=BLUE, markerfmt=".", basefmt=" ")
axes[0].set(xlabel="Time (s)", ylabel="Amplitude", title="s1: an impulse at 0.1 s",
            xlim=(0, timeOfImpulse + 10 * tau_b))
axes[1].plot(t, b, color=GREEN, lw=1.8)
axes[1].set(xlabel=r"Time (out to 10$\tau$)", title=rf"b: blur kernel, $\tau$ = {tau_b} s",
            xlim=(0, timeOfImpulse + 10 * tau_b))
fig.tight_layout()

Convolution of $s_1$ by $b$ is a new function of time, a weighted sum of shifted copies of
$s_1$:

$$s_2(k) \;=\; \sum_{d} s_1(d)\, b(k - d).$$

Notice that $t$ does not appear. Both $k$ and $d$ refer to time, but because we are about to
slide things around, "time" acquires two meanings and it is clearer to keep them apart. The
convolution is evaluated at each output time, which we call $k$; you will come to read $k$ as
an *offset of the blur function*. The signal remains a function of time, but we use a dummy
variable $d$ for it — call it "dime".

There is nothing mysterious about $s_1(d)$: it is the signal, relabelled. The interesting
term is $b(k-d)$. That is $b$ **reflected** about the vertical axis and then **shifted** $k$
steps to the right.

In [ ]:
d = t
ks = np.arange(0, 10)

fig, axes = plt.subplots(10, 1, figsize=(7, 6.5), sharex=True, sharey=True)
for row, k in enumerate(ks):
    axes[row].plot(t[k] - d, b, color=GREEN, lw=1.2)
    axes[row].set_yticks([])
    axes[row].set_ylabel(f"k={k}", fontsize=7, rotation=0, ha="right", va="center")
axes[0].set_xlim(-10 * tau_b, timeOfImpulse + 10 * tau_b)
axes[-1].set_xlabel("Time (d)")
axes[0].set_title("b(k - d) for k = 0 to 9: reflected, then stepped to the right")
fig.tight_layout()

Now generate the convolution one offset at a time. Below, the top row shows $s_1(d)$ in red
(here a 2 Hz sine, so there is something to see) together with the sliding kernel $b(k-d)$ in
green. The middle row shows their pointwise **product**. The bottom row shows the running
**sum** of that product — one number per offset $k$. That sequence of numbers *is* the
convolution.

The original tutorial animates this. Four frozen frames make the same point and can be
compared side by side, which an animation cannot.

In [ ]:
s1 = np.sin(4 * np.pi * t)          # a 2 Hz sine, as in the MATLAB original
b = np.exp(-t / tau_b)

def kernel_at(k):
    '''b(k - d) on the grid d = 0..N-1, zero where k-d is out of range.'''
    out = np.zeros(N)
    idx = k - np.arange(N)                       # this is (k - d) in samples
    ok = (idx >= 0) & (idx < N)
    out[ok] = b[idx[ok]]
    return out

conv_running = np.array([np.sum(s1 * kernel_at(k)) for k in range(N)])

snaps = [5, 15, 30, 60]
fig, axes = plt.subplots(3, 4, figsize=(12, 6), sharex=True)
for col, k in enumerate(snaps):
    kern = kernel_at(k)
    axes[0, col].plot(d, s1, color=RED, lw=1.2)
    axes[0, col].plot(d, kern, color=GREEN, lw=1.6)
    axes[0, col].set_title(f"k = {k}   (t = {t[k]:.2f} s)")
    axes[1, col].plot(d, s1 * kern, color=PURPLE, lw=1.2)
    axes[1, col].fill_between(d, 0, s1 * kern, color=PURPLE, alpha=0.3)
    axes[2, col].plot(t[:k + 1], conv_running[:k + 1], color="k", lw=1.6)
    axes[2, col].plot(t[k], conv_running[k], "o", color=RED, ms=5)
    axes[2, col].set_xlabel("Time (s)")
for row, lab, lo, hi in [(0, "s1(d) and b(k-d)", -1.2, 1.2),
                         (1, "product", -1.2, 1.2),
                         (2, "running sum = s2(k)", -12, 12)]:
    axes[row, 0].set_ylabel(lab, fontsize=8)
    for a in axes[row]:
        a.set_ylim(lo, hi)                    # identical limits across the row
fig.suptitle("Flip, slide, multiply, sum — convolution one output sample at a time")
fig.tight_layout()

That is the whole operation. The impulse-response version — convolving a *delta* with the
exponential — simply reproduces the exponential, smeared to start at the impulse time. The
technical term for smearing is **filtering**. What you just simulated is a sharp click played
through a woofer, or a thin voltage spike seen through an RC circuit.

### II.2 Doing it with `np.convolve`, and the length trap

NumPy will do this for you. But mind the length.

In [ ]:
s2_full  = np.convolve(s1, b, mode="full")
s2_same  = np.convolve(s1, b, mode="same")
s2_valid = np.convolve(s1, b, mode="valid")

print(f"len(s1) = {s1.size}, len(b) = {b.size}")
print(f"  mode='full'  -> {s2_full.size}   (= Ns + Nb - 1; MATLAB conv() default)")
print(f"  mode='same'  -> {s2_same.size}   (centred slice of 'full')")
print(f"  mode='valid' -> {s2_valid.size}   (only fully-overlapping positions)")

# The MATLAB original plots s2(1:length(t)) -- the FIRST N samples of 'full'.
# That is NOT the same slice as mode='same'.
start_same = (b.size - 1) // 2
print(f"\n'same' is full[{start_same}:{start_same + N}]; MATLAB's slice is full[0:{N}]")
print(f"  they differ by {start_same} samples of lag -- "
      f"max|difference| = {np.max(np.abs(s2_full[:N] - s2_same)):.4f}")
print(f"  our by-hand running sum matches full[:N]: "
      f"{np.allclose(conv_running, s2_full[:N])}")

This is the single most common porting bug after indexing. MATLAB's `conv(a,b)` returns the
`'full'` result, and MATLAB code then usually writes `s2(1:length(t))` to get back to the
original length. That keeps the **first** $N$ samples, which preserves causal alignment: the
output at time $t$ depends only on input at times $\le t$. `mode='same'` keeps the **centre**
$N$ samples, which shifts the result earlier by `(len(b)-1)//2` samples. For a causal kernel
like an exponential, `'same'` is the wrong choice and will make your filtered signal appear to
*anticipate* the input.

We reproduce MATLAB's slice — `full[:N]` — throughout, and note it each time.

One more thing the running-sum figure quietly showed: the convolution sum has **no factor of
$dt$**. The discrete sum $\sum_d s_1(d) b(k-d)$ approximates the integral $\int s_1(d)
b(k-d)\,dd$ only up to that factor. Multiplying by $dt$ (or normalizing the kernel to sum to
1) is what keeps the output on the same scale as the input — otherwise a finer sampling rate
silently amplifies your signal.

In [ ]:
s1_imp = np.zeros_like(t)
s1_imp[np.argmin(np.abs(t - timeOfImpulse))] = 1.0
s2_imp = np.convolve(s1_imp, b)[:N]                 # MATLAB slice

gaus_narrow = np.exp(-0.5 * (t - 0.5) ** 2 / 0.012 ** 2)     # SD 12 ms: narrower than the teeth

def circ_conv_zero_lag(x, kern_centred):
    '''Convolve x with a kernel that is drawn centred at t = 0.5 s.

    ifftshift moves the kernel's centre to lag 0, so the output is aligned with
    the input instead of displaced by half a record -- the Part II.5 point,
    used here so the copies sit on top of the teeth that produced them.
    '''
    return np.fft.ifft(np.fft.fft(x) * np.fft.fft(np.fft.ifftshift(kern_centred))).real

s2_narrow = circ_conv_zero_lag(comb10, gaus_narrow)
s2_comb   = circ_conv_zero_lag(comb10, gaus1)                # SD 50 ms: wider than the teeth

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].stem(t, s1_imp, linefmt=RED, markerfmt=".", basefmt=" ", label="s1 (impulse)")
axes[0].plot(t, b, color=GREEN, lw=1.3, label="b (kernel)")
axes[0].plot(t, s2_imp, "--", color=RED, lw=1.8, label="s1 $*$ b")
axes[0].set(xlabel="Time (s)", title="impulse $*$ exponential")
axes[0].legend(fontsize=8)

for ax, kern, out, ttl in [(axes[1], gaus_narrow, s2_narrow, "comb $*$ NARROW Gaussian (SD 12 ms)"),
                           (axes[2], gaus1, s2_comb, "comb $*$ WIDE Gaussian (SD 50 ms)")]:
    ax.stem(t, comb10, linefmt=GREY, markerfmt=".", basefmt=" ", label="comb10")
    ax.plot(t, kern, color="k", lw=1.3, label="Gaussian")
    ax.plot(t, out, "--", color=RED, lw=1.8, label="comb $*$ Gaussian")
    ax.set(xlabel="Time (s)", title=ttl, ylim=(-0.1, 1.45))
    ax.legend(fontsize=7, loc="upper left")
fig.tight_layout()

Convolving with an impulse **reproduces the kernel**, positioned at the impulse. Convolving a
comb with a Gaussian **replicates the Gaussian at every tooth** — clearly visible in the
middle panel, where the Gaussian is narrower than the 0.1 s tooth spacing.

The right-hand panel is the same operation with the original `gaus1` (SD 50 ms), and it looks
completely different: the copies overlap so heavily that they merge into a nearly flat line
with only a small ripple. That is not a different phenomenon, it is the same one — and it is
worth holding on to, because it is exactly the picture of aliasing we will need in Part III.
Replication plus overlap equals irrecoverable mixing.

Both facts are worth remembering: convolution with a delta is the identity operation
(shifted), and convolution with a comb is replication.

> ### Homework question 3
> **(a)** Convolve `pulse1` with `pulse1`. Predict the shape before you run it. (Hint: think
> about how much two rectangles overlap as one slides across the other.)
>
> **(b)** Convolve `gaus1` with a second Gaussian of SD 0.03 s. Measure the SD of the result.
> What is the rule for combining widths under convolution, and why is it not simply additive?
>
> **(c)** Take `b = np.exp(-t/tau_b)` and normalize it so that `b.sum() * dt == 1`. Convolve a
> step function with the normalized and unnormalized kernels. Which one leaves the step height
> unchanged, and why does that matter when you smooth a spike train to get a firing rate?

### II.3 The convolution theorem

Fourier transforms make convolution easy, because **convolution in time is multiplication in
frequency**. If $S_1(\omega)$ is the transform of $s_1(t)$ and $B(\omega)$ the transform of
$b(t)$, then

$$s_2(t) = (s_1 * b)(t) \quad\Longleftrightarrow\quad S_2(\omega) = S_1(\omega)\,B(\omega).$$

That gives a three-step recipe: **(1)** transform both signals, **(2)** multiply them
frequency by frequency (remembering that there is a complex number at each frequency, so this
is complex multiplication — amplitudes multiply and phases *add*), **(3)** inverse transform.

There is one catch, and it is the one from Part 0. The DFT assumes your signal is
**periodic** — that sample $N$ wraps around to sample 0. So `ifft(fft(a) * fft(b))` computes
*circular* convolution, in which the tail that should have run off the end instead reappears
at the beginning. To get the linear convolution `np.convolve` gives you, zero-pad both signals
to length $N_a + N_b - 1$ first. Let's see the difference rather than assert it.

In [ ]:
L = s1.size + b.size - 1

lin_direct  = np.convolve(s1, b, mode="full")
lin_viafft  = np.fft.ifft(np.fft.fft(s1, L) * np.fft.fft(b, L)).real
circ_viafft = np.fft.ifft(np.fft.fft(s1) * np.fft.fft(b)).real     # no padding: circular

print(f"padded FFT product vs np.convolve : max|diff| = "
      f"{np.max(np.abs(lin_direct - lin_viafft)):.3e}")
print(f"unpadded (circular) vs linear[:N] : max|diff| = "
      f"{np.max(np.abs(circ_viafft - lin_direct[:N])):.4f}")
print(f"  ... and that error is concentrated where? argmax at sample "
      f"{int(np.argmax(np.abs(circ_viafft - lin_direct[:N])))} of {N}")

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True, sharey=True)
axes[0].plot(np.arange(L) * dt, lin_direct, color="k", lw=2, label="np.convolve (linear)")
axes[0].plot(np.arange(L) * dt, lin_viafft, "--", color=GREEN, lw=1.5,
             label="ifft(fft*fft), zero-padded to L")
axes[0].axvline(N * dt, color=GREY, ls=":", lw=1)
axes[0].set(ylabel="amplitude", title="Zero-padded: the convolution theorem reproduces np.convolve exactly")
axes[0].legend(fontsize=8)

axes[1].plot(np.arange(L) * dt, lin_direct, color="k", lw=2, label="linear")
axes[1].plot(t, circ_viafft, "--", color=RED, lw=1.5, label="circular (no padding)")
axes[1].axvline(N * dt, color=GREY, ls=":", lw=1)
axes[1].set(xlabel="Time (s)", ylabel="amplitude",
            title="Unpadded: the tail beyond 1 s has wrapped around into the start")
axes[1].legend(fontsize=8)
fig.tight_layout()

The top panel shows the two curves lying on top of each other to $10^{-13}$. The bottom shows
the circular version departing from the linear one at the **beginning** of the record — that
is the part of the convolution that ran past $t = 1$ s, folded back to the front.

When does this matter in practice? Whenever your kernel is not short compared with your data,
or whenever the signal does not start and end near zero. Smoothing a spike train that has
plenty of activity at both ends of the trial, without padding, contaminates the first
milliseconds with the last.

Here is the recipe applied to the impulse and the exponential blur, using the correct padding
and then trimming back to $N$ samples the way MATLAB's `s2(1:length(t))` does.

In [ ]:
newb = np.zeros_like(t)
newb[:b.size] = b               # MATLAB: newb(1:length(b)) = b  (here b is already length N)

B = np.fft.fft(newb, L)
S1 = np.fft.fft(s1_imp, L)
S2 = B * S1                     # multiply frequency by frequency
s2 = np.fft.ifft(S2).real[:N]   # inverse transform, then MATLAB's slice

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t, s2, color=BLUE, lw=2, label="ifft(fft(s1)*fft(b))[:N]")
ax.plot(t, np.convolve(s1_imp, b)[:N], "--", color=RED, lw=1.4, label="np.convolve(...)[:N]")
ax.set(xlabel="Time (s)", ylabel="Amplitude",
       title="Convolution by multiplication in the frequency domain")
ax.legend(fontsize=8)
fig.tight_layout()

print(f"max |difference| between the two routes = "
      f"{np.max(np.abs(s2 - np.convolve(s1_imp, b)[:N])):.3e}")

### II.4 Filtering something realistic

Now a signal worth filtering: the sum of three sines from Part I, corrupted with noise. We
smooth it with a Gaussian kernel and watch what happens in both domains at once.

In [ ]:
s_clean = np.sin(2 * np.pi * 3 * t) + 0.33 * np.sin(2 * np.pi * 9 * t) + 0.2 * np.sin(2 * np.pi * 15 * t)
s_noisy = s_clean + rng.random(N)          # MATLAB: s1 + rand(size(s1)) -- uniform [0,1)
S_noisy = np.fft.fft(s_noisy)

# Gaussian smoothing kernel, ZERO-PHASE: centred at t=0.5 and then ifftshift-ed to index 0.
tau_g = 0.02
g_centred = np.exp(-((t - 0.5) / tau_g) ** 2)
g_centred /= g_centred.sum()               # unit gain at DC, so the mean is preserved
g = np.fft.ifftshift(g_centred)            # <-- what the MATLAB fftshift is for
G = np.fft.fft(g)
S_smooth = G * S_noisy
s_smooth = np.fft.ifft(S_smooth).real

fig, axes = plt.subplots(3, 2, figsize=(10, 7))
axes[0, 0].plot(t, s_noisy, color=GREY, lw=1)
axes[0, 0].plot(t, s_clean, color="k", lw=1.5)
axes[0, 0].set(ylabel="amplitude", title="noisy signal (grey) and the clean one (black)")
axes[0, 1].plot(fax, np.fft.fftshift(np.abs(S_noisy)), color=PURPLE, lw=1)
axes[0, 1].set(ylabel="|FT|", title="its amplitude spectrum")

axes[1, 0].plot(t - 0.5, g_centred, color=GREEN, lw=1.5)
axes[1, 0].set(ylabel="weight", title=f"Gaussian kernel, tau = {tau_g} s (drawn centred)")
axes[1, 1].plot(fax, np.fft.fftshift(np.abs(G)), color=GREEN, lw=1.5)
axes[1, 1].set(ylabel="|G|", title="its transfer function: a low-pass filter")

axes[2, 0].plot(t, s_smooth, color=BLUE, lw=1.5)
axes[2, 0].plot(t, s_clean, color="k", lw=1, alpha=0.6)
axes[2, 0].set(xlabel="Time (s)", ylabel="amplitude", title="smoothed (blue) vs clean (black)")
axes[2, 1].plot(fax, np.fft.fftshift(np.abs(S_smooth)), color=PURPLE, lw=1)
axes[2, 1].set(xlabel="Frequency (Hz)", ylabel="|FT|", title="spectrum after filtering")
for r in (0, 2):
    axes[r, 1].set_ylim(0, 1.05 * np.max(np.abs(S_noisy)))   # comparable panels
    axes[r, 0].set_ylim(-1.6, 2.6)
fig.tight_layout()

print(f"|G| at DC = {np.abs(G[0]):.4f}  (unit gain: the mean survives)")
for f0 in (3, 9, 15, 30, 45):
    print(f"|G| at {f0:2d} Hz = {np.abs(G[int(f0 * tmax)]):.4f}")

Read the right-hand column top to bottom. The noisy spectrum has three clear spikes sitting on
a broad noise floor. The filter's transfer function is a Gaussian that is $\approx 1$ near DC
and falls smoothly with frequency. Their product keeps the low-frequency spikes and
suppresses the high-frequency floor. The printed gains tell you exactly how much each
component survived — and note that the 15 Hz component is attenuated to 0.41, which is why the
recovered trace is noticeably flatter than the original.

Two other things in the bottom-left panel are worth naming rather than ignoring. The smoothed
trace sits about **0.5 above** the clean one, because `rng.random` draws from a uniform
distribution on $[0,1)$ with mean 0.5 — a DC offset, which a low-pass filter by construction
passes untouched. And the very start and end of the smoothed trace curl, because we filtered
circularly with no padding: the end of the record has bled into the beginning, exactly as
Part II.3 warned.

This is the whole idea of filtering: **a filter is a set of multipliers, one per frequency.**

### II.5 Phase, or: why `fftshift` on the kernel matters

The MATLAB original slips in a line, `g2 = fftshift(g1)`, with the comment "see if you can
figure out what the fftshift is doing here". Here is the answer, laid out.

Take a difference-of-Gaussians kernel — a centre-surround filter, the classic model of a
retinal ganglion cell receptive field — built centred at $t = 0.5$ s. Then compare using it
**as built** against using it after shifting its centre to index 0.

In [ ]:
s_box = np.zeros_like(t)
s_box[10:20] = 1.0                    # MATLAB: s1(11:20) = 1  -> Python 10:20
S_box = np.fft.fft(s_box)

tau1, tau2 = tau_b * 0.25, tau_b * 0.5
g1 = (1 / (np.sqrt(2 * np.pi) * tau1) * np.exp(-((t - 0.5) / tau1) ** 2)
      - 1 / (np.sqrt(2 * np.pi) * tau2) * np.exp(-((t - 0.5) / tau2) ** 2))
g2 = np.fft.ifftshift(g1)             # centre moved to index 0  (MATLAB used fftshift; for
                                      #  even N these coincide, ifftshift is the correct one)

G1, G2 = np.fft.fft(g1), np.fft.fft(g2)
SS1, SS2 = G1 * S_box, G2 * S_box
ss1 = np.convolve(s_box, g1)[:N]      # MATLAB's route for the left column
ss2 = np.fft.ifft(SS2).real

print(f"max |G1| = {np.max(np.abs(G1)):.4f},  max |G2| = {np.max(np.abs(G2)):.4f}"
      f"   (identical amplitudes: {np.allclose(np.abs(G1), np.abs(G2))})")
print(f"G1 and G2 differ only in phase: max |G1 - G2| = {np.max(np.abs(G1 - G2)):.4f}")
print(f"centre of mass of |ss1| at t = {np.sum(t * np.abs(ss1)) / np.sum(np.abs(ss1)):.3f} s")
print(f"centre of mass of |ss2| at t = {np.sum(t * np.abs(ss2)) / np.sum(np.abs(ss2)):.3f} s")
print(f"centre of mass of |input| at t = {np.sum(t * s_box) / np.sum(s_box):.3f} s")

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(10, 12), sharex="row")
for col, (kern, Gk, out, SSk, lab) in enumerate(
        [(g1, G1, ss1, SS1, "g1: kernel centred at 0.5 s"),
         (g2, G2, ss2, SS2, "g2 = ifftshift(g1): centred at index 0")]):
    axes[0, col].plot(t, s_box, color=BLUE, lw=1.5)
    axes[0, col].set_title(lab, fontsize=10)
    axes[1, col].plot(fax, np.fft.fftshift(np.abs(S_box)), color=PURPLE, lw=1)
    axes[2, col].plot(t, kern, color=GREEN, lw=1.2)
    axes[3, col].plot(fax, np.fft.fftshift(Gk.real), color=GREEN, lw=1)
    axes[4, col].plot(t, out, color=RED, lw=1.5)
    axes[5, col].plot(fax, np.fft.fftshift(np.abs(SSk)), color=PURPLE, lw=1)
    axes[5, col].set_xlabel("Frequency (Hz)")
for row, lab in enumerate(["input s(t)", "|S(f)|", "kernel g(t)", "Re G(f)",
                           "output (t)", "|S(f)G(f)|"]):
    axes[row, 0].set_ylabel(lab, fontsize=8)
    lo = min(a.get_ylim()[0] for a in axes[row])
    hi = max(a.get_ylim()[1] for a in axes[row])
    for a in axes[row]:
        a.set_ylim(lo, hi)                    # rows share limits so columns are comparable
fig.suptitle("Same kernel, same amplitude spectrum — different phase, very different output",
             y=1.0)
fig.tight_layout()

The two kernels have **identical amplitude spectra** (row 4 is the real part, which does
differ; the printed check confirms $|G_1| = |G_2|$ exactly). They differ only in phase. And
yet the outputs are completely different: the right column's output sits on top of the input,
where it belongs, while the left column's is displaced by half a record — because building
the kernel around $t = 0.5$ s means asking for a filter with a built-in 0.5 s delay.

The printed centres of mass make it quantitative. This is the practical version of a fact
worth internalizing: **amplitude spectra are not the whole story**. Phase carries the timing,
and in a great many signals (speech, images, spike trains) phase carries more of the
recognizable structure than amplitude does.

The correct habit: build a symmetric kernel centred on zero lag, then `np.fft.ifftshift` it
before transforming, or equivalently use `np.convolve` and take the right slice.

> ### Homework question 4
> **(a)** Take the output of the left column, `ss1`, and shift it circularly with
> `np.roll(ss1, -50)`. Does it now match the right column? Explain in terms of the
> $e^{-2\pi i f t_0}$ phase factor.
>
> **(b)** Compute `np.angle(G1)` and `np.angle(G2)` and plot both against frequency. One is a
> straight line through the origin (modulo $2\pi$) and one is not. Which, and why?
>
> **(c)** Scramble the phases of a signal while keeping its amplitude spectrum: take
> `np.fft.fft(s_clean)`, replace each phase with a random one (preserving Hermitian symmetry
> so the inverse is real), and transform back. Does it look anything like the original? What
> does this tell you about which half of the transform carries the "shape"?

---
## Part III. Filters, transfer functions, ringing and aliasing

The original tutorial ends here with a list of topics its authors say they would have covered
"if they paid me more to teach this class". They are too good to leave undone, so we do them.

### III.1 Power spectra

The **power spectrum** of a signal is $|X_k|^2$ — the amplitude spectrum squared, with phase
discarded entirely. It tells you how the signal's variance is distributed across frequency,
and by Parseval it integrates to the total variance. It is the right tool when you care about
*how much* energy is at each frequency and not at all about *when* it arrived — which
describes most analyses of ongoing neural activity, EEG and LFP.

Because the signal is real, we use `rfft`, and we fold the negative-frequency power into the
positive side so that the spectrum sums to the variance. This is a **one-sided** power
spectrum: every bin except DC and Nyquist gets doubled.

In [ ]:
def power_spectrum(x, dt):
    '''One-sided power spectral density of a real signal. Returns (freqs, psd).

    Uses rfft because x is real: the negative frequencies are redundant. The
    factor of 2 folds their power onto the positive side; DC and Nyquist have
    no partner, so they are not doubled.
    '''
    n = x.size
    X = np.fft.rfft(x)
    psd = (np.abs(X) ** 2) / (n ** 2)
    psd[1:-1 if n % 2 == 0 else None] *= 2
    return np.fft.rfftfreq(n, d=dt), psd

Nlong = 4096
tl = np.arange(Nlong) * dt
white = rng.standard_normal(Nlong)

# low-pass it by convolving with a normalized exponential (circularly, via FFT)
kern = np.exp(-np.arange(Nlong) * dt / 0.05)
kern /= kern.sum()
pink = np.fft.ifft(np.fft.fft(white) * np.fft.fft(kern)).real

fw, pw = power_spectrum(white, dt)
fp, pp = power_spectrum(pink, dt)

# Parseval again: the PSD summed over all NON-DC bins is exactly the variance,
# because the DC bin holds the squared mean.
print(f"white noise:    variance = {np.var(white):.8f} ; sum of PSD above DC = {pw[1:].sum():.8f}")
print(f"filtered noise: variance = {np.var(pink):.8f} ; sum of PSD above DC = {pp[1:].sum():.8f}")
print(f"corner frequency 1/(2*pi*tau) = {1 / (2 * np.pi * 0.05):.3f} Hz")

def logsmooth(f, p, nbins=40):
    '''Average the periodogram in logarithmically spaced frequency bins.

    A raw periodogram has ~100% standard error at every bin no matter how long
    the record -- that is the hairiness in the faint traces. Averaging
    neighbouring bins is the cheapest fix (Welch and multitaper are the
    grown-up versions).
    '''
    edges = np.logspace(np.log10(f[1]), np.log10(f[-1]), nbins + 1)
    idx = np.digitize(f, edges)
    fs = np.array([f[idx == i].mean() for i in range(1, nbins + 1) if np.any(idx == i)])
    ps = np.array([p[idx == i].mean() for i in range(1, nbins + 1) if np.any(idx == i)])
    return fs, ps

fig = plt.figure(figsize=(11, 3.6))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.1])
a0 = fig.add_subplot(gs[0, 0]); a1 = fig.add_subplot(gs[1, 0], sharex=a0)
a2 = fig.add_subplot(gs[:, 1])
a0.plot(tl[:400], white[:400], color=GREY, lw=0.8)
a0.set(ylabel="white", title="Two noise processes (own y-scales)")
a1.plot(tl[:400], pink[:400], color=BLUE, lw=1.0)
a1.set(xlabel="Time (s)", ylabel="low-passed")
for f_, p_, colr, lab in [(fw, pw, GREY, "white"), (fp, pp, BLUE, "low-passed")]:
    a2.loglog(f_[1:], p_[1:], color=colr, lw=0.5, alpha=0.25)
    fs, ps = logsmooth(f_[1:], p_[1:])
    a2.loglog(fs, ps, color=colr, lw=2, label=lab)
a2.axvline(1 / (2 * np.pi * 0.05), color="k", ls=":", lw=1)
a2.set(xlabel="Frequency (Hz)", ylabel="power",
       title="Power spectra (faint = raw, bold = log-bin averaged)")
a2.legend(fontsize=8)
fig.tight_layout()

White noise has a flat spectrum — that is the definition of "white", by analogy with light.
The filtered version rolls off above the filter's corner frequency
$f_c = 1/(2\pi\tau) \approx 3.18$ Hz (dotted line), falling as $1/f^2$ in power. Parseval
holds in both cases: the printed PSD sums, taken over every bin **except DC**, equal the
variances exactly. (The DC bin holds the squared mean, which variance excludes.)

Notice the faint raw traces behind the bold averaged ones. A raw periodogram is a wildly
noisy estimate — roughly 100% standard error at every bin, and *that does not improve with a
longer record*, because more data buys you more bins, not better ones. The fix is averaging:
over neighbouring bins (as here), over repeated segments (Welch's method), or over orthogonal
tapers (multitaper). Never report a raw periodogram as if it were an estimate.

A **Bode plot**, while we are naming things, is this same idea applied to a *system* rather
than a signal: gain (usually in decibels) and phase, both plotted against log frequency.

### III.2 Impulse responses and transfer functions

Suppose you play a click through a speaker and record what comes out. The recorded waveform
is the speaker's **impulse response**, $h(t)$. Its Fourier transform $H(f)$ is the
**transfer function**.

Why is that useful? Because *if* the system is (i) **linear** and (ii) **time-invariant** —
the two tenets of linear systems theory — then the impulse response tells you the response to
*any* input at all. Any input is a sum of scaled, shifted impulses (that is what a discrete
signal *is*, from Part I.2); linearity says the responses add; time-invariance says each
shifted impulse produces the same response, shifted. Adding up the shifted, scaled copies of
$h$ **is** convolution:

$$y(t) = (x * h)(t) \qquad\Longleftrightarrow\qquad Y(f) = X(f)\, H(f).$$

So one measurement — the response to a click — predicts the response to music. Both
assumptions are approximations for any real speaker and gross approximations for any neuron,
but they are the right place to start, and the LN and Wiener–Volterra models you will meet
later are the systematic ways of relaxing them.

Let's *measure* a transfer function from input and output data, rather than being told it.

In [ ]:
h_true = np.exp(-np.arange(Nlong) * dt / 0.05)
h_true /= h_true.sum()

x_in  = rng.standard_normal(Nlong)                                   # white probe
y_out = np.fft.ifft(np.fft.fft(x_in) * np.fft.fft(h_true)).real      # circular, so exact

X_in, Y_out = np.fft.fft(x_in), np.fft.fft(y_out)
H_est = Y_out / X_in                     # works because the probe is white: no zero bins
h_est = np.fft.ifft(H_est).real

f_pos = np.fft.rfftfreq(Nlong, d=dt)
H_true_f = np.fft.rfft(h_true)
H_est_f  = H_est[:f_pos.size]

print(f"max |H_est - H_true| over all bins = "
      f"{np.max(np.abs(H_est - np.fft.fft(h_true))):.3e}")
print(f"max |h_est - h_true| in the time domain = {np.max(np.abs(h_est - h_true)):.3e}")
print(f"gain at DC   = {np.abs(H_true_f[0]):.4f}")
i3 = int(np.argmin(np.abs(f_pos - 3.18)))
print(f"gain at {f_pos[i3]:.2f} Hz = {np.abs(H_true_f[i3]):.4f}   "
      f"(the corner, 1/(2*pi*tau) = {1 / (2 * np.pi * 0.05):.2f} Hz, where gain ~ 0.707)")
print(f"phase at the corner  = {np.degrees(np.angle(H_true_f[i3])):+.1f} deg "
      f"(continuous-time first-order filter: -45 deg)")
print(f"phase minimum        = {np.degrees(np.angle(H_true_f)).min():+.1f} deg at "
      f"{f_pos[int(np.argmin(np.angle(H_true_f)))]:.1f} Hz")
print(f"phase at Nyquist     = {np.degrees(np.angle(H_true_f[-1])):+.1f} deg "
      f"(a DISCRETE filter must be real at Nyquist)")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].plot(tl[:60], h_true[:60], color="k", lw=2, label="true h(t)")
axes[0].plot(tl[:60], h_est[:60], "--", color=RED, lw=1.4, label="estimated")
axes[0].set(xlabel="Time (s)", ylabel="weight", title="Impulse response")
axes[0].legend(fontsize=8)
axes[1].loglog(f_pos[1:], np.abs(H_true_f[1:]), color="k", lw=2)
axes[1].loglog(f_pos[1:], np.abs(H_est_f[1:]), "--", color=RED, lw=1)
axes[1].axvline(1 / (2 * np.pi * 0.05), color=GREY, ls=":")
axes[1].set(xlabel="Frequency (Hz)", ylabel="|H|", title="Gain (Bode magnitude)")
axes[2].semilogx(f_pos[1:], np.degrees(np.angle(H_true_f[1:])), color="k", lw=2)
axes[2].axvline(1 / (2 * np.pi * 0.05), color=GREY, ls=":")
axes[2].set(xlabel="Frequency (Hz)", ylabel="phase (deg)", title="Phase (Bode phase)")
fig.tight_layout()

The estimate is exact here, to $\sim 10^{-14}$, because we generated the data by circular
convolution with no noise and probed with white noise, so no frequency bin was left
unexcited. In real life you divide by small numbers wherever your probe had little power, and
the estimate blows up. The standard fix is the **cross-spectral estimator**
$\hat H = S_{xy}/S_{xx}$ averaged over repeats, which is the same expression with the
numerator and denominator smoothed first.

Now look at the phase plot, and read the printed numbers alongside it, because this is a place
where the textbook picture and the discrete reality part company.

A *continuous-time* first-order low-pass filter has phase $-45°$ exactly at the corner and
approaches $-90°$ asymptotically. Our filter is a **discrete** geometric kernel, and its
transfer function is

$$H(f) \;=\; \frac{1-a}{1 - a\,e^{-2\pi i f\,dt}}, \qquad a = e^{-dt/\tau}.$$

At low frequency this matches the continuous formula, and indeed the gain at the corner comes
out at 0.709 — the textbook $1/\sqrt2$. But the phase reaches only $-39.4°$ there rather than
$-45°$, bottoms out at $-55°$ near 10 Hz, and then **returns to exactly $0°$ at Nyquist**, because
$e^{-2\pi i f dt} = -1$ there, which makes $H$ purely real. There is no way around this: any
real-valued kernel has a real transfer function at Nyquist.

The lesson is not that the figure is wrong; it is that **discrete filters are not sampled
continuous filters**, and the discrepancy grows as you approach Nyquist. If you need a
digital filter that matches an analogue design, use a proper mapping (`scipy.signal.bilinear`)
rather than sampling the impulse response and hoping.

The phase lag itself is the quantitative content of "the speaker lags the signal", and it is
why filtering a spike train with a causal kernel delays your rate estimate.

### III.3 Why you cannot build a perfect filter: ringing

Filtering is attenuating frequencies. So why not design a filter by simply *specifying* the
amplitude spectrum you want — say, 1 between 10 and 20 Hz and 0 everywhere else?

Because of the pulse–sinc pair from Part I.7, plus the **duality** of the Fourier transform:
if $f(t) \leftrightarrow F(\omega)$ is a transform pair, then so is $F(t) \leftrightarrow
f(\omega)$. A rectangle in *frequency* therefore has a **sinc** in *time* — an impulse
response that rings forever, and in both directions, so it is not even causal. You cannot
have a filter that is sharp in frequency and compact in time. It is the uncertainty principle
again.

In [ ]:
Nf = 512
tf = np.arange(Nf) * dt
ff = np.fft.fftfreq(Nf, d=dt)

H_ideal = ((np.abs(ff) >= 10) & (np.abs(ff) <= 20)).astype(float)   # brick-wall band-pass
h_ideal = np.fft.fftshift(np.fft.ifft(H_ideal).real)                # centred for viewing
lag = (np.arange(Nf) - Nf // 2) * dt

# what happens if we truncate that impulse response to something implementable?
for width in (0.05, 0.20):
    keep = np.abs(lag) <= width
    h_trunc = np.where(keep, h_ideal, 0.0)
    H_trunc = np.abs(np.fft.fft(np.fft.ifftshift(h_trunc)))
    inband = (np.abs(ff) >= 10) & (np.abs(ff) <= 20)
    stop = (np.abs(ff) < 8) | (np.abs(ff) > 22)
    print(f"kernel truncated to +/-{width:.2f} s ({keep.sum():3d} taps): "
          f"pass band gain {H_trunc[inband].min():.3f} to {H_trunc[inband].max():.3f} "
          f"(want 1.000), worst stop-band leak {H_trunc[stop].max():.3f} (want 0.000)")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].plot(np.fft.fftshift(ff), np.fft.fftshift(H_ideal), color="k", lw=2)
axes[0].set(xlabel="Frequency (Hz)", ylabel="gain", title="The filter we asked for",
            ylim=(-0.1, 1.3))
axes[1].plot(lag, h_ideal, color=RED, lw=1.2)
axes[1].set(xlabel="Lag (s)", ylabel="weight",
            title="The impulse response it implies: a sinc")
axes[1].set_xlim(-1.0, 1.0)
for width, colr in [(0.05, BLUE), (0.20, GREEN)]:
    keep = np.abs(lag) <= width
    H_trunc = np.abs(np.fft.fft(np.fft.ifftshift(np.where(keep, h_ideal, 0.0))))
    axes[2].plot(np.fft.fftshift(ff), np.fft.fftshift(H_trunc), color=colr, lw=1.3,
                 label=f"truncated to $\\pm${width:.2f} s")
axes[2].plot(np.fft.fftshift(ff), np.fft.fftshift(H_ideal), color="k", lw=1, ls=":",
             label="ideal")
axes[2].set(xlabel="Frequency (Hz)", ylabel="gain", title="What you actually get",
            ylim=(-0.1, 1.3))
axes[2].legend(fontsize=8)
fig.tight_layout()

Neither truncation gives you what you asked for, and they fail in different ways. The short
11-tap kernel is too blunt: its pass band never reaches 1 and it leaks badly into the stop
band. The longer 41-tap kernel has much sharper edges and ten times less leakage, but now the
pass band **overshoots to 1.12** and rings on both sides of each edge. That overshoot is the
**Gibbs phenomenon**, and its height does not shrink as you lengthen the kernel — it just gets
narrower. Adding taps does not remove it.

(The printed pass-band *minimum* of about 0.5 in both cases is a different, expected thing: it
occurs in the bins exactly at 10 and 20 Hz, where the ideal filter is discontinuous, and any
truncation converges there to the midpoint of the jump, 0.5. That is Gibbs again, from the
other side.)

This trade-off is the entire subject of filter design, and it is why practical filters
(Butterworth, Chebyshev, elliptic, FIR windowed designs) all have gently sloping edges instead
of brick walls, and why FIR designs taper the kernel rather than chopping it off.

The same fact, read the other way round: whenever you **window** data — cut a 500 ms epoch out
of a long recording — you have multiplied by a rectangle, and therefore convolved your
spectrum with a sinc. That is spectral leakage. Tapered windows (Hann, Hamming, discrete
prolate spheroidal sequences) trade a wider main lobe for far smaller side lobes.

> ### Homework question 5
> **(a)** Repeat the truncation experiment with a Hann-tapered kernel
> (`h_trunc * np.hanning(...)` over the kept region). How do in-band ripple and stop-band
> leakage change?
>
> **(b)** The ideal impulse response above is symmetric in lag, so it is **non-causal** — the
> output at time $t$ depends on input at times *later* than $t$. Why is that fatal for
> real-time filtering but perfectly acceptable for offline analysis of a recorded trace?
>
> **(c)** Real-world instances of the amplitude-spectrum picture: a cheap speaker muffling
> high frequencies, resonance and timbre in a musical instrument, blur from an imperfect lens.
> For each, sketch $|H(f)|$ and say what it does to a broadband input.

### III.4 Sampling and aliasing

Recall from Part I.2 that a sampled signal is a continuous signal **multiplied** by a comb.
And recall the convolution theorem: convolution in time is multiplication in frequency. The
theorem runs both ways, so **multiplication in time is convolution in frequency**:

$$x(t)\,c(t) \quad\Longleftrightarrow\quad \frac{1}{N}\,\bigl(X * C\bigr)(f).$$

The transform of a comb is a comb (Part I.7, row 7). So sampling **replicates the entire
spectrum at every multiple of the sampling frequency**. If the original spectrum is narrower
than half the sampling rate, the copies do not touch and you can throw them away. If it is
wider, the copies overlap, and the overlapping part appears at a frequency where it does not
belong. That is **aliasing**, and once it has happened no amount of processing will undo it.

Here is the demonstration. Start with a square wave at 27 Hz — deliberately chosen to be
below our 50 Hz Nyquist, so the *fundamental* is safe, but a square wave has harmonics at
$3f$, $5f$, $7f$, ... and those are not.

In [ ]:
ysq = np.sign(np.cos(2 * np.pi * 27 * t - np.pi / 7))
Ysq = np.fft.fft(ysq)

top = np.argsort(-np.abs(Ysq))[:8]
print("Strongest components of the 27 Hz square wave, as sampled at 100 Hz:")
for k in sorted(set(np.abs(np.fft.fftfreq(N, dt)[top]))):
    kk = int(k * tmax)
    print(f"   {k:5.1f} Hz   |X| = {np.abs(Ysq[kk]):7.3f}")
print("\nA square wave's true harmonics are at 27, 81, 135, 189, ... Hz.")
for har in (27, 81, 135, 189):
    a = har % samplingRate
    a = a if a <= nyq else samplingRate - a
    print(f"   {har:4d} Hz folds to {a:5.1f} Hz")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(t, ysq, color=BLUE, lw=1.2, drawstyle="steps-post")
axes[0].set(xlabel="Time (s)", ylabel="amplitude", title="27 Hz square wave, sampled at 100 Hz",
            xlim=(0, 0.3), ylim=(-1.3, 1.3))
axes[1].plot(fax, np.fft.fftshift(np.abs(Ysq)), color=PURPLE, lw=1.2)
axes[1].set(xlabel="Frequency (Hz)", ylabel="|FT|", title="Its spectrum — already aliased")
fig.tight_layout()

The spectrum shows four pairs of peaks, at 11, 19, 27 and 35 Hz. Only **27 Hz is real**. The
other three are the 3rd, 5th and 7th harmonics — at 81, 135 and 189 Hz — folded back below
Nyquist by the sampling, exactly as the printed arithmetic says. The square wave was aliased
the moment we sampled it at 100 Hz, before we did anything else.

(The low, grassy floor of amplitude 2–5 across the whole axis is a separate effect: 27 Hz does
not divide the 100 Hz sample grid, so the square wave's edges land between samples and jitter
by up to one sample from cycle to cycle. That jitter is broadband. It is a small reminder that
you rarely get a spectrum as clean as the textbook ones in Part I.)

Now sample it *again*, much more coarsely: keep only one point in ten, using our 10 Hz comb.

In [ ]:
ysampled = ysq * comb10
Ysamp = np.fft.fft(ysampled)

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
axes[0, 0].plot(t, ysq, color=GREY, lw=0.8, drawstyle="steps-post")
axes[0, 0].plot(t[::10], ysq[::10], "o-", color=BLUE, lw=1.6, ms=6)
axes[0, 0].set(ylabel="amplitude", title="square wave (grey) and its 10 Hz strobe samples (blue)",
               ylim=(-1.4, 1.4))
axes[0, 1].plot(fax, np.fft.fftshift(np.abs(Ysq)), color=PURPLE, lw=1.2)
axes[0, 1].set(ylabel="|FT|", title="spectrum before strobing")
axes[1, 0].plot(t[::10], ysq[::10], "o-", color=BLUE, lw=1.6, ms=6)
axes[1, 0].set(xlabel="Time (s)", ylabel="amplitude",
               title="what the strobe sees: a slow, irregular waveform", ylim=(-1.4, 1.4))
axes[1, 1].plot(fax, np.fft.fftshift(np.abs(Ysamp)), color=RED, lw=1.2)
axes[1, 1].set(xlabel="Frequency (Hz)", ylabel="|FT|",
               title="spectrum after strobing: periodic every 10 Hz")
for a in (axes[0, 1], axes[1, 1]):
    a.set_ylim(0, 1.05 * np.max(np.abs(Ysq)))     # comparable panels
fig.tight_layout()

print(f"|FT| at  3 Hz: before strobing {np.abs(Ysq[3]):7.4f}   after {np.abs(Ysamp[3]):7.4f}")
print(f"|FT| at  7 Hz: before strobing {np.abs(Ysq[7]):7.4f}   after {np.abs(Ysamp[7]):7.4f}")
print(f"|FT| at 17 Hz: before strobing {np.abs(Ysq[17]):7.4f}   after {np.abs(Ysamp[17]):7.4f}")
print(f"|FT| at 27 Hz: before strobing {np.abs(Ysq[27]):7.4f}   after {np.abs(Ysamp[27]):7.4f}")
print(f"\nAfter strobing, is the spectrum periodic with period 10 Hz (10 bins)?  "
      f"{np.allclose(Ysamp, np.roll(Ysamp, 10))}")

# Multiplication in time <-> circular convolution in frequency, divided by N.
Ccomb = np.fft.fft(comb10)
circconv = np.array([np.sum(Ysq * Ccomb[(k - np.arange(N)) % N]) for k in range(N)]) / N
print(f"Is the strobed spectrum (1/N) * circular convolution of the two spectra?  "
      f"{np.allclose(Ysamp, circconv)}   (max diff {np.max(np.abs(Ysamp - circconv)):.2e})")

This is the extreme case, and it is worth reading carefully. The strobed spectrum is
**exactly periodic with period 10 Hz** — the printed check confirms it, and you can see it in
the red panel, where the same pattern repeats ten times across the axis. Every bin has the
same amplitude as the bin 10 Hz away from it.

That periodicity *is* aliasing, stated as strongly as it can be stated: after strobing at 10
Hz, the frequencies 3, 13, 23, 33 and 43 Hz are **numerically indistinguishable**. The 27 Hz
peak that stood at $|X| = 63.7$ before strobing has been smeared into ten identical copies of
$6.47$ each. We put no 3 Hz or 7 Hz component into this signal, and yet there they are; they
are copies of the 27 Hz component (and of the folded harmonics) shifted by multiples of the
strobe rate. $27 - 10 - 10 - 10 = -3$, so 27 Hz reappears at 3 Hz; $27 - 20 = 7$ Hz likewise.

The last printed line verifies the mechanism directly rather than asserting it: the strobed
spectrum *is* the circular convolution of the original spectrum with the comb's spectrum,
divided by $N$. The comb's spectrum is a set of spikes every 10 Hz (Part I.7, row 7), and
convolving with a set of spikes replicates — which is exactly the "comb $*$ wide Gaussian"
picture from Part II.2, now happening in the frequency domain.

This is why every data-acquisition system has an **anti-aliasing filter** in hardware, before
the digitizer. Once the copies have overlapped there is no software fix, because there is no
way to tell which copy a given frequency came from.

> ### Homework question 6
> **(a)** Change the square wave to 27 Hz *sine* rather than a square wave and repeat the
> strobing. Where do the aliases land now, and why are there fewer of them?
>
> **(b)** Low-pass filter `ysq` below 5 Hz *before* strobing (an anti-aliasing filter), then
> strobe. Compare the resulting spectrum with the unfiltered one. What has been lost, and what
> has been saved?
>
> **(c)** A wagon wheel in an old film appears to turn backwards. Frames arrive at 24 Hz.
> Using the aliasing arithmetic above, find the wheel speeds (in spokes per second) at which
> it appears stationary, and at which it appears to turn slowly backwards.
>
> **(d)** Sketch, on paper, the spectrum of a signal band-limited to 40 Hz after sampling at
> 100 Hz, and then after sampling at 60 Hz. At what sampling rate do the copies first touch?

> ### Homework question 7
> These are the "additional topics" the original tutorial leaves as an exercise. They are
> conceptual; write a paragraph on each.
>
> **(a)** Develop better intuitions about filters as multiplication in the frequency domain.
> Find real-world examples: the muffling of a cheap speaker, resonance and timbre in musical
> instruments, blurring of an image by an imperfect lens.
>
> **(b)** What is a Bode plot? What is a power spectrum? How do they differ in what they
> describe — and in what they throw away?
>
> **(c)** What is an impulse response function, and what is it good for? If you play a click
> through a speaker, it comes out a little less sharp. How does the shape of that output
> predict what the speaker will do with music? Answer in terms of convolution and Fourier
> transforms, and state the two tenets of linear systems theory that the argument depends on.
>
> **(d)** Given that filtering is attenuation of frequencies, what goes wrong when you try to
> design a filter by specifying its amplitude spectrum? Why can you not build one that passes
> 10–20 Hz and cuts everything else? (Hint 1: transform pairs are dual — if $f(t)
> \leftrightarrow F(\omega)$, then $F(t) \leftrightarrow f(\omega)$. Hint 2: apply that to
> the pulse $\leftrightarrow$ sinc pair.)

---
## Summary

1. The **DFT** re-expresses $N$ samples in a new orthogonal basis of complex exponentials:
   $X_k = \sum_n x_n e^{-2\pi i k n/N}$, with the $1/N$ on the inverse. It is a rotation, not
   a distortion — the signal is unchanged, only its description is.

2. Each frequency needs **two** numbers, amplitude and phase, or equivalently a cosine weight
   (real part) and a sine weight (imaginary part). The bookkeeping works out exactly:
   $N$ real samples $\rightarrow$ $N$ independent real numbers in the transform, with the
   negative frequencies fixed by **Hermitian symmetry** $X_{N-k} = \overline{X_k}$.

3. Learn the **transform pairs** by heart: Gaussian $\leftrightarrow$ Gaussian (reciprocal
   widths), pulse $\leftrightarrow$ sinc, exponential $\leftrightarrow$ $1/f$ roll-off,
   delta $\leftrightarrow$ flat, comb $\leftrightarrow$ comb. Everything else in this tutorial
   is a consequence of one of them.

4. **Convolution** — flip, slide, multiply, sum — is filtering. In the frequency domain it is
   plain multiplication, $S_2 = S_1 B$: a filter is a set of multipliers, one per frequency.

5. That equivalence has a catch: the DFT's multiplication is **circular**. Zero-pad to
   $N_a + N_b - 1$ to get the linear convolution, or the end of your signal contaminates the
   beginning.

6. **Phase matters.** Two kernels with identical amplitude spectra can produce completely
   different outputs. Build symmetric kernels around zero lag and `ifftshift` them.

7. Sampling multiplies by a comb, which convolves the spectrum with a comb, which
   **replicates** it every $1/dt$ Hz. If the signal is not band-limited below Nyquist, the
   copies overlap and the extra frequencies fold back as **aliases** that cannot be removed
   afterwards. Filter before you digitize.

8. Sharp in frequency means spread out in time, and vice versa. This is why perfect filters
   ring, why short data windows leak, and why a narrow Gaussian has a wide spectrum. It is one
   fact wearing several hats.

### A porting checklist

If you take one practical thing from this tutorial, take Part 0. Before you believe any
spectrum you have computed:

- Check **Parseval**. It catches every normalization error.
- Check the **convolution theorem** with explicit zero-padding. It catches circular-wrap bugs.
- Feed in a **sinusoid of known frequency** and confirm the peak lands in the bin you expect.
  It catches every axis and off-by-one error.

Three cheap tests, and between them they catch essentially all of the ways a spectrum goes
quietly wrong.

### Further reading

- Bracewell, R. N. *The Fourier Transform and Its Applications*, 3rd ed. The standard
  reference; the pictorial dictionary of transform pairs in the early chapters is worth the
  price on its own.
- Oppenheim, A. V. & Schafer, R. W. *Discrete-Time Signal Processing*. The authority on the
  discrete case, including everything about sampling, aliasing and filter design.
- Percival, D. B. & Walden, A. T. *Spectral Analysis for Physical Applications*. Where to go
  for real spectral estimation — tapers, multitaper methods, confidence intervals.
- Rieke, F., Warland, D., de Ruyter van Steveninck, R. & Bialek, W. *Spikes: Exploring the
  Neural Code*. Linear systems ideas applied to neural coding, including reverse correlation.
- `scipy.signal` — `butter`, `filtfilt`, `firwin`, `welch`, `spectrogram`. Once you understand
  what these do, use them rather than rolling your own.